In [1]:
pip install requests pandas

In [ ]:
import requests
import pandas as pd
import os
import time
import json
from datetime import datetime
from typing import List, Dict, Any, Optional, Tuple

SNAPSHOT_URL = "https://hub.snapshot.org/graphql"

# ==========================
# ✅ CONFIG
# ==========================
SPACES_FILE = "data/processed/selected_spaces_follower_knee.json"

OUT_DIR = "data/processed/votes_441_per_space"
SPACE_CSV_DIR = os.path.join(OUT_DIR, "spaces")
PROGRESS_DIR = os.path.join(OUT_DIR, "progress_by_space")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(SPACE_CSV_DIR, exist_ok=True)
os.makedirs(PROGRESS_DIR, exist_ok=True)

FAIL_LOG = os.path.join(OUT_DIR, "failures.log")

PROPOSAL_BATCH = 100
VOTES_PAGE_SIZE = 1000  # 起始页大小，超时会自动降到 500/250/200
MIN_VOTES_PAGE_SIZE = 200

# 网络/限流参数
SLEEP_BETWEEN_CALLS = 1.6     # 429 多或超时多就 1.8~2.2
MAX_RETRIES = 6
BACKOFF_BASE = 2.0
HTTP_TIMEOUT = (10, 90)       # (connect_timeout, read_timeout)

# 小规模测试开关（建议先跑 5 个 space、每个 3 个 proposal）
MAX_SPACES = None
MAX_PROPOSALS_PER_SPACE = None

# 可选：先抓 followersCount 并按重要性排序（会多一些请求，但更符合“先抓最重要的”）
SORT_BY_FOLLOWERS = True
# ==========================


# ---------- Utilities ----------
def safe_filename(name: str) -> str:
    """Make a filesystem-safe filename for a space id."""
    keep = []
    for ch in name:
        if ch.isalnum() or ch in ("-", "_", "."):
            keep.append(ch)
        else:
            keep.append("_")
    return "".join(keep)


def space_csv_path(space: str) -> str:
    return os.path.join(SPACE_CSV_DIR, f"{safe_filename(space)}.csv")


def log_failure(tag: str, msg: str):
    with open(FAIL_LOG, "a", encoding="utf-8") as f:
        f.write(f"[{tag}] {msg}\n")


# ---------- Robust GraphQL ----------
def post_gql(query: str, variables: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """
    GraphQL POST with:
    - retry + exponential backoff for 429/5xx/timeouts
    - stop-fast on 400 (query invalid)
    """
    payload = {"query": query}
    if variables:
        payload["variables"] = variables

    last_err = None
    last_status = None
    last_text = None

    for attempt in range(MAX_RETRIES):
        try:
            r = requests.post(SNAPSHOT_URL, json=payload, timeout=HTTP_TIMEOUT)
            last_status = r.status_code
            last_text = r.text

            # 400: query 不合法，直接抛出
            if r.status_code == 400:
                raise RuntimeError(f"HTTP 400 Bad Request: {r.text[:800]}")

            # 429/5xx: 需要重试
            if r.status_code in (429, 500, 502, 503, 504):
                raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")

            r.raise_for_status()

            data = r.json()
            if "errors" in data:
                raise RuntimeError(f"GQL errors: {str(data['errors'])[:800]}")

            return data["data"]

        except Exception as e:
            last_err = e

            # 400 不重试
            if isinstance(e, RuntimeError) and "HTTP 400" in str(e):
                print("\n========== SNAPSHOT 400 BAD REQUEST ==========")
                print("Response (first 800 chars):", (last_text or "")[:800])
                print("Query (first 800 chars):", query[:800])
                print("=============================================\n")
                raise

            wait = (BACKOFF_BASE ** attempt) + 0.2 * attempt
            print(f"⚠️ 请求失败（attempt {attempt+1}/{MAX_RETRIES}）：{e} | backoff {wait:.1f}s")
            time.sleep(wait)

    raise RuntimeError(
        f"❌ GraphQL failed after {MAX_RETRIES} retries. "
        f"last_status={last_status}, last_error={repr(last_err)}"
    )


# ---------- Load 441 spaces ----------
def load_selected_space_ids(spaces_file: str) -> List[str]:
    with open(spaces_file, "r", encoding="utf-8") as f:
        obj = json.load(f)

    ids = obj.get("selected_space_ids", [])
    if not isinstance(ids, list):
        raise ValueError("selected_space_ids is not a list in SPACES_FILE")

    seen = set()
    out = []
    for s in ids:
        if isinstance(s, str) and s and s not in seen:
            seen.add(s)
            out.append(s)

    return out


# ---------- Progress per space ----------
def progress_path_for_space(space: str) -> str:
    return os.path.join(PROGRESS_DIR, f"progress_{safe_filename(space)}.json")


def load_progress(space: str) -> set:
    path = progress_path_for_space(space)
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return set(json.load(f))
    return set()


def save_progress(space: str, completed: set):
    path = progress_path_for_space(space)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(sorted(list(completed)), f, ensure_ascii=False, indent=2)


# ---------- Snapshot helpers ----------
def fetch_space_followers(space: str) -> Optional[int]:
    q = f"""
    {{
      space(id: "{space}") {{
        followersCount
      }}
    }}
    """
    data = post_gql(q)
    sp = data.get("space")
    if not sp:
        return None
    fc = sp.get("followersCount")
    try:
        return int(fc) if fc is not None else None
    except:
        return None


def fetch_all_proposals(space: str) -> List[Dict[str, Any]]:
    all_props = []
    skip = 0

    print(f"\n🚀 开始抓取 Space={space} 的全部 proposals ...")
    while True:
        q = f"""
        {{
          proposals(first: {PROPOSAL_BATCH}, skip: {skip},
            where: {{ space_in: ["{space}"] }},
            orderBy: "created", orderDirection: desc
          ) {{
            id
            title
            body
            choices
            author
            state
            created
          }}
        }}
        """
        data = post_gql(q)
        batch = data.get("proposals", [])
        print(f"📦 {space} proposals: {skip + 1} ~ {skip + len(batch)}")

        if not batch:
            break

        all_props.extend(batch)
        if len(batch) < PROPOSAL_BATCH:
            break

        skip += PROPOSAL_BATCH
        time.sleep(SLEEP_BETWEEN_CALLS)

    print(f"✅ {space} proposals 总数：{len(all_props)}")
    return all_props


def fetch_all_votes_for_proposal(proposal_id: str) -> List[Dict[str, Any]]:
    """
    Cursor pagination via created_lt (desc) to bypass skip<=5000 limit.
    Add dynamic page_size reduction on timeout/instability.
    """
    all_votes = []
    created_lt = int(time.time()) + 10  # 32-bit safe
    seen = set()

    page_size = VOTES_PAGE_SIZE

    while True:
        q = f"""
        {{
          votes(first: {page_size},
            where: {{ proposal: "{proposal_id}", created_lt: {created_lt} }},
            orderBy: "created", orderDirection: desc
          ) {{
            voter
            choice
            vp
            created
          }}
        }}
        """

        try:
            data = post_gql(q)
        except Exception as e:
            msg = str(e).lower()
            # 超时/连接波动：降低 page_size 再试，减少响应体
            if ("read timed out" in msg or "timeout" in msg or "connectionpool" in msg) and page_size > MIN_VOTES_PAGE_SIZE:
                page_size = max(MIN_VOTES_PAGE_SIZE, page_size // 2)
                print(f"⚠️ 超时/波动：降低 votes page_size → {page_size} 后重试")
                time.sleep(2.0)
                continue
            raise

        batch = data.get("votes", [])
        if not batch:
            break

        for v in batch:
            key = (v.get("voter"), v.get("created"))
            if key in seen:
                continue
            seen.add(key)
            all_votes.append(v)

        last_created = batch[-1]["created"]
        if last_created >= created_lt:  # safety
            break
        created_lt = last_created

        if len(batch) < page_size:
            break

        time.sleep(SLEEP_BETWEEN_CALLS)

    return all_votes


def compute_majority_choice(votes: List[Dict[str, Any]]) -> Optional[int]:
    counter: Dict[int, int] = {}
    for v in votes:
        c = v.get("choice")
        if isinstance(c, int):
            counter[c] = counter.get(c, 0) + 1
        elif isinstance(c, dict):
            for k, val in c.items():
                try:
                    kk = int(k)
                except:
                    continue
                if val and val > 0:
                    counter[kk] = counter.get(kk, 0) + 1
    if not counter:
        return None
    return max(counter.items(), key=lambda x: x[1])[0]


def append_rows_to_space_csv(space: str, rows: List[Dict[str, Any]]):
    """Append rows into one CSV per space."""
    if not rows:
        return
    path = space_csv_path(space)
    df = pd.DataFrame(rows)
    header = not os.path.exists(path)
    df.to_csv(path, mode="a", index=False, header=header)


# ---------- MAIN ----------
def main():
    spaces = load_selected_space_ids(SPACES_FILE)
    print(f"✅ 从 selected_space_ids 读取 spaces 数量：{len(spaces)}")

    if MAX_SPACES is not None:
        spaces = spaces[:MAX_SPACES]
        print(f"🧪 MAX_SPACES={MAX_SPACES} → 本次只跑前 {len(spaces)} 个 spaces")

    # followers sort (optional)
    space_items: List[Tuple[str, Optional[int]]] = [(s, None) for s in spaces]
    if SORT_BY_FOLLOWERS:
        print("🔎 获取 followersCount 并排序（None 放最后）...")
        tmp = []
        for i, s in enumerate(spaces, 1):
            try:
                fc = fetch_space_followers(s)
            except Exception as e:
                fc = None
                log_failure("FOLLOWERS_FAIL", f"{s} | {repr(e)}")
            tmp.append((s, fc))
            if i % 25 == 0:
                print(f"   ... followers fetched {i}/{len(spaces)}")
            time.sleep(0.2)
        tmp.sort(key=lambda x: (x[1] is None, -(x[1] or 0)))
        space_items = tmp
        print("✅ 排序后前 5 个：", space_items[:5])

    for idx, (space, followers) in enumerate(space_items, 1):
        print("\n" + "=" * 90)
        print(f"🏷️  Space [{idx}/{len(space_items)}]: {space} | followersCount={followers}")
        print("=" * 90)

        completed = load_progress(space)
        print(f"📌 已完成 proposals 数：{len(completed)}（{progress_path_for_space(space)}）")
        print(f"📄 输出 CSV：{space_csv_path(space)}")

        # space-level try
        try:
            proposals = fetch_all_proposals(space)
        except Exception as e:
            log_failure("SPACE_FAIL", f"{space} | {repr(e)}")
            print(f"❌ Space 抓 proposals 失败（已记录 failures.log），跳过：{e}")
            continue

        if MAX_PROPOSALS_PER_SPACE is not None:
            proposals = proposals[:MAX_PROPOSALS_PER_SPACE]
            print(f"🧪 MAX_PROPOSALS_PER_SPACE={MAX_PROPOSALS_PER_SPACE} → 本次只跑 {len(proposals)} 个 proposals")

        for i, p in enumerate(proposals, 1):
            pid = p["id"]
            if pid in completed:
                continue

            title = (p.get("title") or "")[:80]
            print(f"\n🔍 [{space}] Proposal {i}/{len(proposals)}：{title}")

            try:
                votes = fetch_all_votes_for_proposal(pid)
                print(f"🧾 votes 数：{len(votes)}")

                # 无 votes 也标记完成
                if not votes:
                    completed.add(pid)
                    save_progress(space, completed)
                    continue

                total_vp = sum(v.get("vp", 0) for v in votes)
                if total_vp == 0:
                    completed.add(pid)
                    save_progress(space, completed)
                    continue

                majority = compute_majority_choice(votes)
                if majority is None:
                    completed.add(pid)
                    save_progress(space, completed)
                    continue

                rows = []
                for v in votes:
                    vp = v.get("vp", 0)
                    vp_ratio = vp / total_vp if total_vp else 0

                    choice_used = None
                    aligned = False
                    c = v.get("choice")

                    if isinstance(c, int):
                        choice_used = c
                        aligned = (c == majority)
                    elif isinstance(c, dict):
                        choice_used = json.dumps(c, ensure_ascii=False)
                        aligned = (str(majority) in c and c[str(majority)] > 0)

                    rows.append({
                        "Space": space,
                        "FollowersCount": followers,
                        "Proposal ID": pid,
                        "Proposal Title": p.get("title", ""),
                        "Proposal Body": (p.get("body") or "").replace("\n", " ").replace("\r", " ")[:1000],
                        "Created Time": datetime.utcfromtimestamp(p["created"]).isoformat(),
                        "Voter": v.get("voter"),
                        "Choice": choice_used,
                        "Voting Power": vp,
                        "VP Ratio (%)": round(vp_ratio * 100, 4),
                        "Is Whale": vp_ratio > 0.05,
                        "Aligned With Majority": aligned,
                        "Vote Timestamp": datetime.utcfromtimestamp(v["created"]).isoformat()
                    })

                append_rows_to_space_csv(space, rows)

                completed.add(pid)
                save_progress(space, completed)

                print(f"✅ 写入 {len(rows)} 行 | space 完成 proposals={len(completed)}")
                time.sleep(SLEEP_BETWEEN_CALLS)

            except Exception as e:
                log_failure("PROPOSAL_FAIL", f"{space} | {pid} | {repr(e)}")
                print(f"❌ Proposal 失败（已记录 failures.log），继续：{e}")
                continue

    print("\n🎉 全部完成（或已尽可能完成）！")
    print(f"📁 Space CSV 目录：{SPACE_CSV_DIR}")
    print(f"📄 failures：{FAIL_LOG}")
    print(f"📌 progress_dir：{PROGRESS_DIR}")


if __name__ == "__main__":
    main()


✅ 从 selected_space_ids 读取 spaces 数量：441
🔎 获取 followersCount 并排序（None 放最后）...
   ... followers fetched 25/441
   ... followers fetched 50/441
   ... followers fetched 75/441
   ... followers fetched 100/441
   ... followers fetched 125/441
⚠️ 请求失败（attempt 1/6）：HTTP 429: {"error":"unauthorized","error_description":"too many requests, refer to https://docs.snapshot.box/tools/api/api-keys#limits"} | backoff 1.0s
⚠️ 请求失败（attempt 2/6）：HTTP 429: {"error":"unauthorized","error_description":"too many requests, refer to https://docs.snapshot.box/tools/api/api-keys#limits"} | backoff 2.2s
⚠️ 请求失败（attempt 3/6）：HTTP 429: {"error":"unauthorized","error_description":"too many requests, refer to https://docs.snapshot.box/tools/api/api-keys#limits"} | backoff 4.4s
⚠️ 请求失败（attempt 4/6）：HTTP 429: {"error":"unauthorized","error_description":"too many requests, refer to https://docs.snapshot.box/tools/api/api-keys#limits"} | backoff 8.6s
⚠️ 请求失败（attempt 5/6）：HTTP 429: {"error":"unauthorized","error_descrip

🧾 votes 数：6747
✅ 写入 6747 行 | space 完成 proposals=29

🔍 [stgdao.eth] Proposal 30/155：[Fast Track] HEP-1: Hydra Expansion Program
🧾 votes 数：3384
✅ 写入 3384 行 | space 完成 proposals=30

🔍 [stgdao.eth] Proposal 31/155：Stargate Deployment on Abstract
🧾 votes 数：14554
✅ 写入 14554 行 | space 完成 proposals=31

🔍 [stgdao.eth] Proposal 32/155：Stargate Deployment on Hedera
🧾 votes 数：13735
✅ 写入 13735 行 | space 完成 proposals=32

🔍 [stgdao.eth] Proposal 33/155：ETH Pool on Manta Pacific
🧾 votes 数：11927
✅ 写入 11927 行 | space 完成 proposals=33

🔍 [stgdao.eth] Proposal 34/155：Stargate Deployment on Conflux
🧾 votes 数：10614
✅ 写入 10614 行 | space 完成 proposals=34

🔍 [stgdao.eth] Proposal 35/155：Final Data RFP Vote - Choice of Artemis Options
🧾 votes 数：10735
✅ 写入 10735 行 | space 完成 proposals=35

🔍 [stgdao.eth] Proposal 36/155：Stargate Deployment to Gnosis Chain
🧾 votes 数：15493
✅ 写入 15493 行 | space 完成 proposals=36

🔍 [stgdao.eth] Proposal 37/155：Absolutely thrilling news for all Stargate users!
🧾 votes 数：10
✅ 写入 10 行 | sp